# 빅데이터 분석기사 실기 모의고사 7회차

## 작업형 3유형 · 통계 분석 — One-way ANOVA + Levene + Welch's ANOVA (3문항 × 10점 = 30점)

**출제 범위**: 일원분산분석 / Levene 등분산성 / Welch's ANOVA (등분산 위배 대안)
**사용 라이브러리**: `scipy.stats` (pingouin 대신 scipy 직접 계산 — 폐쇄형 시험 환경 호환)
**유의수준**: α = 0.05

> 📌 **본 회차 3유형 핵심 — ANOVA 흐름 3단계**
> 1단계: One-way ANOVA 수행 + 효과 크기 (η²)
> 2단계: 등분산 가정 점검 (Levene)
> 3단계: 가정 위배 시 Welch's ANOVA 로 robust 확인
>
> 이 흐름은 의학·심리학·사회과학 논문의 표준 보고 방식입니다.

---

## 데이터 로드 (heart_disease_uci.csv)

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

BASE = 'https://raw.githubusercontent.com/ashram68/bigdata-analytics-exam/main/data/exam-07/'
df = pd.read_csv(BASE + 'heart_disease_uci.csv')

# 분석 변수 결측 제거
df_clean = df.dropna(subset=['oldpeak', 'cp'])
print(df_clean.shape)

# cp 그룹 4개
for cp_type in df_clean['cp'].unique():
    g = df_clean[df_clean['cp'] == cp_type]['oldpeak']
    print(f'{cp_type:20s}: n={len(g):3d}, mean={g.mean():.4f}, std={g.std():.4f}')

---

## 문제 7-3-1 (10점) — One-way ANOVA: cp → oldpeak

`cp` (흉통 유형, 4개 그룹: typical angina / asymptomatic / non-anginal / atypical angina) 에 따라
`oldpeak` (운동 유발 ST 우울증) 평균이 다른지 **일원분산분석** 으로 검정하시오.
(`oldpeak` 결측치는 제거, α=0.05)

### 가설
- H₀: 4개 cp 그룹의 oldpeak 평균이 모두 동일
- H₁: 적어도 한 그룹의 oldpeak 평균이 다름

### 수행
`scipy.stats.f_oneway()` + **η²(eta-squared) 효과 크기** 직접 계산
(공식: `η² = SS_between / SS_total`)

### 출력 형식
1. **F-통계량** (소수점 둘째 자리)
2. **η²** (eta-squared, 소수점 넷째 자리)
3. **검정 결론**: "기각" 또는 "채택"

> 💡 **η² 해석**: 0.01 작음 / 0.06 중간 / 0.14 큼

In [ ]:
# 여기에 코드를 작성하세요.

# groups = [df_clean[df_clean['cp']==cp]['oldpeak'].values for cp in df_clean['cp'].unique()]
# f_stat, p_value = stats.f_oneway(*groups)
#
# # η² 직접 계산
# all_data = np.concatenate(groups)
# grand_mean = np.mean(all_data)
# ss_between = sum(len(g) * (np.mean(g) - grand_mean)**2 for g in groups)
# ss_total   = np.sum((all_data - grand_mean)**2)
# eta_sq     = ss_between / ss_total
#
# print(f'F  = {f_stat:.2f}')
# print(f'eta_sq = {eta_sq:.4f}')
# print('기각' if p_value < 0.05 else '채택')


---

## 문제 7-3-2 (10점) — Levene 등분산성 검정 (ANOVA 가정 점검)

[7-3-1] 에서 사용한 cp 4그룹의 `oldpeak` 분산이 동일한지 **Levene 검정** 으로 확인하시오. (α=0.05)

### 가설
- H₀: 4개 그룹의 분산이 모두 동일 (등분산)
- H₁: 적어도 한 그룹의 분산이 다름

### 수행
`scipy.stats.levene(*groups)`

### 출력 형식
1. **Levene W-통계량** (소수점 둘째 자리)
2. **p-value** (소수점 넷째 자리)
3. **검정 결론**: "기각" 또는 "채택"

> 💡 **의미**: 기각되면 ANOVA 의 등분산 가정이 위배 → Welch's ANOVA 사용 권장.

In [ ]:
# 여기에 코드를 작성하세요.

# W_stat, p_value = stats.levene(*groups)
# print(f'W = {W_stat:.2f}')
# print(f'p = {p_value:.4f}')
# print('기각' if p_value < 0.05 else '채택')


---

## 문제 7-3-3 (10점) — Welch's ANOVA (등분산 위배 시 대안)

[7-3-2] 에서 등분산 가정이 위배되었음을 확인했으므로, **Welch's ANOVA** 로 재검정하여
[7-3-1] 의 결론이 robust 한지 확인하시오. (α=0.05)

### 가설
[7-3-1] 과 동일 (cp 그룹별 oldpeak 평균 차이)

### 수행 (scipy 로 직접 계산 — Welch-Satterthwaite 자유도 보정)
본 모의고사는 폐쇄형 시험 환경(외부 패키지 설치 불가) 을 가정하므로, `pingouin` 대신 `scipy` 로 직접 계산합니다.

### 출력 형식
1. **Welch's F-통계량** (소수점 둘째 자리)
2. **p-value** (소수점 넷째 자리)
3. **검정 결론**: "기각" 또는 "채택"

> 💡 일반 ANOVA(F=32.59) 와 Welch's ANOVA(F=48.53) 의 결론을 비교하여 robust성 확인.

In [ ]:
# 여기에 코드를 작성하세요.

# # Welch's ANOVA — scipy 직접 계산
# k     = len(groups)
# means = [np.mean(g) for g in groups]
# vars_ = [np.var(g, ddof=1) for g in groups]
# ns    = [len(g) for g in groups]
#
# # 가중치 w_i = n_i / s_i²
# weights = [n/v for n,v in zip(ns, vars_)]
# sum_w   = sum(weights)
# weighted_mean = sum(w*m for w,m in zip(weights, means)) / sum_w
#
# # F 분자
# numer = sum(w*(m - weighted_mean)**2 for w,m in zip(weights, means)) / (k-1)
#
# # F 분모 (Welch-Satterthwaite)
# sum_term = sum((1 - w/sum_w)**2 / (n-1) for w,n in zip(weights, ns))
# denom    = 1 + (2*(k-2)/(k**2-1)) * sum_term
#
# F_welch = numer / denom
# df1     = k - 1
# df2     = (k**2 - 1) / (3 * sum_term)
# p_welch = 1 - stats.f.cdf(F_welch, df1, df2)
#
# print(f"Welch's F = {F_welch:.2f}")
# print(f'p = {p_welch:.4f}')
# print('기각' if p_welch < 0.05 else '채택')


---

## 풀이 종료 안내

- **3유형 합격선**: 30점 만점 기준 18점 이상

### 본 회차 3유형 핵심 — ANOVA 의 3가정

1. **정규성 (Normality)**: 각 그룹이 정규분포 따름. 검정 `stats.shapiro()`. 위배 시 → Kruskal-Wallis (`stats.kruskal`).
2. **등분산성 (Homogeneity of Variance)**: 모든 그룹의 분산이 동일. 검정 `stats.levene()`. 위배 시 → Welch's ANOVA.
3. **독립성 (Independence)**: 관측치 간 독립. 연구 설계로 보장.

### 3가지 ANOVA 비교

| 항목 | 일반 ANOVA | Welch's ANOVA | Kruskal-Wallis |
|---|---|---|---|
| 등분산 가정 | 필요 | **불필요** | 불필요 |
| 정규성 가정 | 필요 | 필요 | **불필요** |
| Python 함수 | `f_oneway()` | scipy 직접 계산 | `kruskal()` |
| 통계량 | F | F (보정) | H |

---

*폐쇄형 시험 환경 적응 모의고사 - 빅데이터 분석기사 실기 (Heart Disease UCI 데이터 기준)*